# SPIKE-046: teacher runs in Colab

The runner (`jenab-scenarios`) and the teacher (`gemma4:31b` on Ollama) run in this VM. Each run is a real Jenab run: the real prompt, tools, SQLite database and asserts. Passing runs become training examples.

- No keys: the repository is public, and a local Ollama needs none. `.env` never comes here.
- Reports and the export go to Drive after each chunk of scenarios. After a disconnect, run the cells again from the top: finished chunks are skipped.
- Order: **1** setup, **2** scripted smoke (no GPU), **3** Ollama and the teacher, **4** the 10-run check, **5** the training sets, **6** the export.
- Disconnect the runtime when you are done: units are spent while it is connected.

In [ ]:
# 1. Settings. Change these, then run every cell in order.
import os

REPO = 'https://github.com/amir-saatchi/Jenab.git'
BRANCH = 'main'                # the branch with SPIKE-046's sets and export
GO_VERSION = '1.27.3'
TEACHER = 'gemma4:31b'
CONTEXT = 8192                 # 8k on an L4 (24 GB); more on an A100
PARALLEL = 1                   # runs at a time; 2-4 on an A100 with OLLAMA_NUM_PARALLEL to match
REPS = 1                       # runs of each scenario
CHUNK = 10                     # scenarios per chunk; a chunk is saved to Drive when it ends
USE_DRIVE = True

WORK = '/content/jenab'
OUT = '/content/drive/MyDrive/jenab-spike046' if USE_DRIVE else '/content/spike046'
os.environ.update({'PATH': '/usr/local/go/bin:' + os.environ['PATH'], 'CGO_ENABLED': '0'})

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)
print('output:', OUT)

In [ ]:
# 1. Go, the repository and the runner. CGO_ENABLED=0: the SQLite driver is pure Go.
import subprocess, pathlib

def sh(cmd, check=True):
    # Runs cmd and prints its output as it comes (Colab shows a child's output only through Python).
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    code = p.wait()
    if check and code != 0:
        raise SystemExit(f'failed ({code}): {cmd}')
    return code

if not pathlib.Path('/usr/local/go/bin/go').exists() or GO_VERSION not in subprocess.run('/usr/local/go/bin/go version', shell=True, capture_output=True, text=True).stdout:
    sh(f'curl -fsSL https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz -o /tmp/go.tgz')
    sh('rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz')
sh('go version')
if pathlib.Path(WORK).exists():
    sh(f'git -C {WORK} fetch --depth 1 origin {BRANCH} && git -C {WORK} checkout -q FETCH_HEAD')
else:
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO} {WORK}')
os.chdir(WORK)
sh('git log -1 --format="%h %s"')
sh('go build -o /content/jenab-scenarios ./cmd/jenab-scenarios')
JS = '/content/jenab-scenarios'
sh(f'{JS} check scenarios | grep sql-')

In [ ]:
# 2. Scripted smoke, no GPU and no model: every training set plays its _offline.yaml
# (the reference answers) through the real app, then the export runs on the reports.
# Checks the Linux build, the sets and the export before any GPU time is spent.
import glob

TRAIN = sorted(os.path.basename(d) for d in glob.glob('scenarios/sql-train-*'))
HELDOUT = sorted(os.path.basename(d) for d in glob.glob('scenarios/sql-heldout-*'))
print('training sets:', TRAIN)
sh('rm -rf /content/smoke')
for s in TRAIN:
    sh(f'{JS} run -examples -parallel 4 -model scripted:scenarios/{s}/_offline.yaml -out /content/smoke/{s} scenarios/{s} | tail -1')
sh(f'{JS} export -out /content/smoke/export /content/smoke')
sh('wc -l /content/smoke/export/*.jsonl')

In [ ]:
# 3. Ollama and the teacher, as an ordinary ollama connection at 127.0.0.1:11434.
# The runner sends num_ctx from the context in models.yaml.
import time, urllib.request

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
if sh('which ollama', check=False) != 0:
    sh('curl -fsSL https://ollama.com/install.sh | sh')

def ollama_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        return True
    except Exception:
        return False

if not ollama_up():
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(CONTEXT), OLLAMA_KEEP_ALIVE='-1', OLLAMA_NUM_PARALLEL=str(PARALLEL))
    subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True, env=env)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
sh(f'ollama pull {TEACHER}')
sh('ollama list')

pathlib.Path('scenarios/models.yaml').write_text(f'''# Colab: the teacher on the VM's own Ollama. No key.
providers:
  - name: colab
    kind: ollama
    base_url: http://127.0.0.1:11434/
    models:
      - {{id: "{TEACHER}", context: {CONTEXT}}}
''')
MODEL = f'colab/{TEACHER}'
print('model:', MODEL)

In [ ]:
# 4. The 10-run check: 10 scenarios of the first training set, with the time per run.
# Use it to size the larger runs: runs per hour, and units per hour from Colab's resources panel.
import json, statistics, re

def scenario_ids(s):
    ids = []
    for p in sorted(glob.glob(f'scenarios/{s}/*.yaml')):
        if os.path.basename(p).startswith('_'):
            continue
        m = re.search(r'^id: (\S+)$', pathlib.Path(p).read_text(encoding='utf-8'), re.M)
        if m:
            ids.append(m.group(1))
    return ids

def newest_report(d):
    files = sorted(f for f in glob.glob(f'{d}/*.json') if re.search(r'/\d{8}-\d{6}\.json$', f))
    return json.load(open(files[-1], encoding='utf-8')) if files else None

def times(rep):
    runs = [r for r in rep['runs'] if not r.get('skipped')]
    secs = [r['seconds'] for r in runs]
    passed = sum(1 for r in runs if not r.get('error') and all(a['status'] != 'fail' or a.get('test') == 'info' for a in r['asserts']))
    return runs, secs, passed

first = TRAIN[0]
ids = scenario_ids(first)[:10]
d = f'{OUT}/check10'
t0 = time.time()
sh(f'{JS} run -examples -model {MODEL} -parallel {PARALLEL} -out {d} -scenario {",".join(ids)} scenarios/{first}')
wall = time.time() - t0
runs, secs, passed = times(newest_report(d))
print(f'{len(runs)} runs, {passed} passed, wall {wall/60:.1f} min')
print(f'per run: mean {statistics.mean(secs):.0f} s, median {statistics.median(secs):.0f} s, max {max(secs):.0f} s')
print(f'about {3600 * len(runs) / wall:.0f} runs an hour at -parallel {PARALLEL}')
total = sum(len(scenario_ids(s)) for s in TRAIN) * REPS
print(f'the training sets: {total} runs, about {total * wall / len(runs) / 3600:.1f} h')

In [ ]:
# 5. The training sets, in chunks of CHUNK scenarios. Each chunk writes its report and
# examples to Drive and a DONE file; a chunk with DONE is skipped, so after a disconnect
# this cell resumes where it stopped. A chunk cut halfway runs again from its start.
log = open(f'{OUT}/runs.log', 'a', encoding='utf-8')
for s in TRAIN:
    ids = scenario_ids(s)
    for i in range(0, len(ids), CHUNK):
        d = f'{OUT}/reports/{s}/chunk{i // CHUNK:02d}'
        if os.path.exists(f'{d}/DONE'):
            continue
        t0 = time.time()
        code = sh(f'{JS} run -examples -model {MODEL} -reps {REPS} -parallel {PARALLEL} -out {d} -scenario {",".join(ids[i:i + CHUNK])} scenarios/{s} | tail -3', check=False)
        rep = newest_report(d)
        if code != 0 or rep is None:
            print(f'{s} chunk {i // CHUNK}: the runner failed ({code}); run the cell again')
            break
        runs, secs, passed = times(rep)
        line = f'{time.strftime("%Y-%m-%d %H:%M")} {s} chunk{i // CHUNK:02d}: {passed}/{len(runs)} passed, {time.time() - t0:.0f} s'
        print(line)
        log.write(line + '\n'); log.flush()
        pathlib.Path(f'{d}/DONE').write_text(line + '\n')
log.close()

In [ ]:
# 6. The export: every passing run of the training reports in Drive, as chat.jsonl and
# subagents.jsonl. It refuses held-out reports, and fails on personal IDs.
sh(f'{JS} export -out {OUT}/export {OUT}/reports')
sh(f'wc -l {OUT}/export/*.jsonl')
sh(f'tail -20 {OUT}/runs.log', check=False)

## After the runs

- Note in the ticket: the runs, the passes, the failures and why (the reports' Markdown files say), the GPU hours and the units.
- Then disconnect: **Runtime → Disconnect and delete runtime**.